# Fase 3 – Etiquetado Manual y Basado en Reglas de Logs Web

Este notebook implementa un **etiquetado semiautomático robusto** combinando:

- Isolation Forest (anomalías estadísticas)
- Reglas basadas en patrones de ataques web (firmas)
- Correcciones manuales documentadas

Objetivo: generar un dataset de **alta calidad** para entrenamiento supervisado.

## 1. Carga del dataset procesado

In [19]:
import pandas as pd

# Ajusta el nombre del archivo según tu pipeline
df = pd.read_csv('access_log_master_manual_lab_if_score.csv')
df1 = pd.read_csv('../data/target/access_logs_manual_if_scores.csv')
df.head()

,ip_client,timestamp,status,size,user_agent,method,url,protocol,anomaly,status_category,...,url__digit_count,url__letter_count,url__count_special_characters,url__is_encoded,url__unusual_character_ratio,if_prediction,if_score,attack_patterns,signature_flag,severity
0,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/06/b2ap3_lar...,HTTP/2.0,0,200,...,10,52,12,0,0.094595,1,0.042052,[],0,medium
1,47.128.121.63,2025-11-25 00:00:16-05:00,200,675,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/plugins/elementor/assets/lib/font-...,HTTP/2.0,0,200,...,4,63,16,0,0.120482,1,0.030259,[],0,medium
2,47.128.121.63,2025-11-25 00:00:16-05:00,200,264,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/elementor/css/pos...,HTTP/2.0,0,200,...,15,43,12,0,0.128571,1,0.041807,[],0,medium
3,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/05/73192906_...,HTTP/2.0,0,200,...,57,26,13,0,0.072917,1,0.029720,[],0,medium
4,47.128.121.63,2025-11-25 00:00:16-05:00,206,500,Mozilla/5.0 (Linux; Android 5.0) AppleWebKit/5...,GET,/wp-content/uploads/sites/15/2021/07/PNG-image...,HTTP/2.0,0,200,...,16,33,12,0,0.114754,1,0.045362,[],0,medium


In [3]:
df['if_prediction'].value_counts()


if_prediction
 1    896441
-1    158165
Name: count, dtype: int64

In [4]:
df1['if_prediction'].value_counts()

if_prediction
 1    1024714
-1      29639
Name: count, dtype: int64

In [20]:
import re

RCE_REGEX = re.compile(
    r"""
    (                                   # Grupo principal
        # Web shells conocidos
        (shell|cmd|backdoor|c99|r57|wso|b374k|uploadshell|webshell)\.(php|asp|jsp)
        |
        # Ejecución de comandos
        (;|\||&&|\$\(|`)\s*
        (ls|cat|id|whoami|uname|wget|curl|nc|bash|sh)
        |
        # Binarios del sistema Linux
        (/bin/|/usr/bin/|/etc/passwd|/proc/self)
        |
        # Windows / PowerShell
        (cmd\.exe|powershell|net\s+user|system32)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)
TRAVERSAL_REGEX = re.compile(
    r"""
    (
        (\.\./){2,}                    # ../ repetido
        |
        (%2e%2e%2f|%2e%2e/|\.%2e/)     # Traversal codificado
        |
        (/etc/passwd|/etc/shadow|/root/|/boot/)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

SENSITIVE_FILES_REGEX = re.compile(
    r"""
    (
        # Configuración y secretos
        (\.env|wp-config\.php|config\.php)
        |
        # Backups
        (\.bak|\.old|\.backup|~$)
        |
        # Repositorios
        (\.git/|\.svn/|\.hg/)
        |
        # Dumps / comprimidos
        (\.zip|\.tar|\.gz|\.sql)$
    )
    """,
    re.IGNORECASE | re.VERBOSE
)
INJECTION_REGEX = re.compile(
    r"""
    (
        # XSS
        (<script|onerror=|onload=|javascript:)
        |
        # SQL Injection
        ('|%27)\s*(or|and|union|select|sleep|benchmark)
        |
        # PHP Injection
        (\$\{|\$\(|eval\(|base64_decode)
        |
        # Código ofuscado
        (base64,|%3c%3fphp)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)
XMLRPC_REGEX = re.compile(
    r"""
    (
        /xmlrpc\.php
        |
        system\.multicall
        |
        wp\.getUsersBlogs
    )
    """,
    re.IGNORECASE | re.VERBOSE
)




def is_category_a_attack(url: str) -> bool:
    if not isinstance(url, str):
        return False

    return any(
        regex.search(url)
        for regex in [
            RCE_REGEX,
            TRAVERSAL_REGEX,
            SENSITIVE_FILES_REGEX,
            INJECTION_REGEX,
            XMLRPC_REGEX
        ]
    )


In [21]:
STATIC_EXTENSIONS = (
    '.css', '.js', '.png', '.jpg', '.jpeg', '.gif',
    '.svg', '.woff', '.woff2', '.ttf', '.ico', '.map', '.pdf'
)

ADMIN_SENSITIVE_PATHS = (
    '/wp-admin/',
    '/wp-login.php',
    '/xmlrpc.php',
    '/phpmyadmin',
    '/manager',
    '/cpanel'
)

SUSPICIOUS_UA = re.compile(
    r"(python-requests|curl|go-http-client|java)",
    re.IGNORECASE
)

CRAWLER_UA = re.compile(
    r"(googlebot|bingbot|baiduspider|applebot|yandex)",
    re.IGNORECASE
)

## 2. Reglas de detección basadas en patrones (firmas)

In [22]:

# def detect_attack_patterns(url, method,user_agent):
#     url = str(url).lower()
#     patterns = []
#     admin_keywords = r"\b(admin|wp-admin|phpmyadmin|manager|cpanel|login)\b"

#     suspicious_agents = r"(python-requests|curl|Go-http-client|Java|bot|spider|crawler|^$)"

#     # OGNL / RCE (Apache Struts)
#     if re.search(r"(\$\{|%24%7b).*context", url):
#         patterns.append('ognl_rce')

#     # SQL Injection
#     if re.search(r"('|%27)(\s|%20)*(or|union|select|sleep|benchmark)", url):
#         patterns.append('sqli')

#     # XSS
#     if re.search(r"(<script|onerror=|onload=|%3cscript)", url):
#         patterns.append('xss')

#     # Path Traversal
#     if re.search(r"(\.\./|%2e%2e)", url):
#         patterns.append('path_traversal')

#     # Command Injection
#     if re.search(r"(;|\||&&)(ls|cat|whoami|id)", url):
#         patterns.append('command_injection')

#     # # WordPress probing
#     # if (
#     #   '/wp-content/plugins/' in url and
#     #   (
#     #       method in ['POST', 'PUT'] or
#     #       '?' in url or
#     #       url.endswith('.php')
#     #   )
#     #   ):
#     #    patterns.append('wp_attack')
      
#     if(is_category_a_attack(url)):
#         patterns.append('attack')
       

#     # Admin / login probing
#     if ( re.search(admin_keywords, url, re.IGNORECASE) 
#         and re.search(suspicious_agents, user_agent, re.IGNORECASE) ):
#       patterns.append("admin_probe")
      

#     # Method probing
#     if method in ['OPTIONS', 'TRACE', 'CONNECT']:
#         patterns.append('method_probe')

#     return patterns

def detect_attack_patterns(url, method, user_agent):
    url = str(url).lower()
    user_agent = str(user_agent).lower()
    patterns = []

    # ------------------------------------------------------------------
    # 0. Ignorar assets estáticos (reduce FP masivamente)
    # ------------------------------------------------------------------
    if url.endswith(STATIC_EXTENSIONS):
        return patterns

    # ------------------------------------------------------------------
    # 1. Categoría A – Ataque confirmado (determinístico)
    # ------------------------------------------------------------------
    if is_category_a_attack(url):
        patterns.append('attack_confirmed')
        return patterns  # cortocircuito intencional

    # ------------------------------------------------------------------
    # 2. Ataques WordPress críticos
    # ------------------------------------------------------------------
    if url == '/wp-admin/admin-ajax.php' and method == 'POST':
        patterns.append('wp_ajax_attack')
        return patterns

    if url == '/xmlrpc.php':
        patterns.append('xmlrpc_attack')
        return patterns

    # ------------------------------------------------------------------
    # 3. Inyecciones explícitas
    # ------------------------------------------------------------------
    if re.search(r"('|%27)\s*(or|union|select|sleep|benchmark)", url):
        patterns.append('sqli')
    
    if re.search(r"(union(\s+all)?\s+select|extractvalue|updatexml|sleep\(|benchmark\(|xp_cmdshell|information_schema)", url, re.IGNORECASE):
      patterns.append('sqli')
    
    # Detecta SQLi avanzada con CAST, CASE, operadores booleanos y ofuscación
    if re.search(
        r"(cast\s*\(|"
        r"case\s+when|"
        r"\|\||"                     # concatenación típica de PostgreSQL
        r"::(text|int|numeric)|"     # type casting PostgreSQL
        r"/\*.*?\*/|"                # comentarios ofuscados
        r"and\s+\d+\s*=\s*\d+|"      # boolean-based
        r"--\s*\+?-?)",              # terminación de consulta
        url,
        re.IGNORECASE | re.DOTALL
    ):
        patterns.append("sqli_advanced")



    if re.search(r"(<script|onerror=|onload=|%3cscript)", url):
        patterns.append('xss')

    if re.search(r"(\.\./|%2e%2e)", url):
        patterns.append('path_traversal')

    if re.search(r"(;|\||&&)(ls|cat|whoami|id)", url):
        patterns.append('command_injection')

    # ------------------------------------------------------------------
    # 4. Admin probing (versión estricta, sin FP)
    # ------------------------------------------------------------------
    is_admin_path = any(p in url for p in ADMIN_SENSITIVE_PATHS)

    if is_admin_path:
        # crawlers NO deberían estar aquí nunca
        if CRAWLER_UA.search(user_agent):
            patterns.append('admin_probe')

        # clientes no humanos ejecutando acciones
        elif SUSPICIOUS_UA.search(user_agent):
            patterns.append('admin_probe')

        # métodos no normales
        elif method not in ['GET']:
            patterns.append('admin_probe')

    # ------------------------------------------------------------------
    # 5. Method probing
    # ------------------------------------------------------------------
    if method in ['OPTIONS', 'TRACE', 'CONNECT']:
        patterns.append('method_probe')

    return patterns

df['attack_patterns'] = df.apply(lambda r: detect_attack_patterns(r['url'], r['method'],r["user_agent"]), axis=1)
df['signature_flag'] = df['attack_patterns'].apply(lambda x: 1 if len(x) > 0 else 0)

df[['url', 'attack_patterns', 'signature_flag']].head()

,url,attack_patterns,signature_flag
0,/wp-content/uploads/sites/15/2021/06/b2ap3_lar...,[],0
1,/wp-content/plugins/elementor/assets/lib/font-...,[],0
2,/wp-content/uploads/sites/15/elementor/css/pos...,[],0
3,/wp-content/uploads/sites/15/2021/05/73192906_...,[],0
4,/wp-content/uploads/sites/15/2021/07/PNG-image...,[],0


In [8]:
def show(df_review,num):
    labels = {}
    i=0
    for idx, row in df_review.iterrows():
        i=i+1
        print("="*120)
        print(f"Index: {idx}")
        print(f"IP: {row['ip_client']}")
        print(f"Timestamp: {row['timestamp']}")
        print(f"Method: {row['method']}  Status: {row['status']}")
        print(f"URL:\n{row['url']}")
        print(f"User-Agent:\n{row['user_agent']}")
        print(f"Anomaly: {row['anomaly']}")
        print(f"if_prediction: {row['if_prediction']}")

        print(f"Patterns detectados: {row['attack_patterns']}")
        

        if(i==num):break
        


In [23]:
df["attack_patterns"].value_counts()
# filtro = df[df['attack_patterns'].apply(lambda x: 'attack_confirmed' in x) & (df['anomaly'] != 1)]
# cantidad = filtro.shape[0]



attack_patterns
[]                            1015872
[attack_confirmed]              20250
[wp_ajax_attack]                 8987
[admin_probe]                    6766
[sqli_advanced]                  1576
[method_probe]                    562
[sqli, sqli_advanced]             535
[sqli]                             46
[path_traversal]                    6
[sqli, sqli_advanced, xss]          5
[xss]                               1
Name: count, dtype: int64

In [118]:
df['anomaly'].value_counts()

anomaly
 0    492237
-1    475987
 1     86382
Name: count, dtype: int64

In [6]:
filtro = df[df['attack_patterns'].apply(lambda x: 'attack_confirmed' in x) & (df['anomaly'] != 1)]
cantidad = filtro.shape[0]
print(cantidad)


0


In [120]:
filtro = df[df['attack_patterns'].apply(lambda x: 'attack_confirmed' in x) & (df['anomaly'] == -1)]
cantidad = filtro.shape[0]
print(cantidad)

0


In [106]:
# Seleccionar las filas que cumplen la condición y actualizar 'anomaly' a 1
df.loc[
    df['attack_patterns'].apply(lambda x: 'attack_confirmed' in x) & (df['anomaly'] == -1),
    'anomaly'
] = 1


In [108]:
df['anomaly'].value_counts()


anomaly
 0    492237
-1    475987
 1     86382
Name: count, dtype: int64

In [109]:
df.to_csv(
    'access_log_master_manual_lab_if_score.csv',
    index=False
)

In [ ]:
is_forest_fails = df[df["attack_patterns"].apply(lambda x: "attack_confirmed" in x) & (df['if_prediction'] == 1)]
cantidad = is_forest_fails.shape[0]
print(cantidad)
show(is_forest_fails,10)


In [ ]:
wp_ajax_attack = df[df["attack_patterns"].apply(lambda x: "wp_ajax_attack" in x)]
show(wp_ajax_attack,10)


In [ ]:
wp_ajax_attack = df[df["attack_patterns"].apply(lambda x: "wp_ajax_attack" in x) & (df['anomaly']!=1)]
cant=wp_ajax_attack.shape[0]
print(cant)
show(wp_ajax_attack,20)


In [ ]:
admin_probe = df[df["attack_patterns"].apply(lambda x: "admin_probe" in x)]
show(admin_probe,10)

In [ ]:
admin_probe = df[df["attack_patterns"].apply(lambda x: "admin_probe" in x) & (df['anomaly']!=1)]
cant=admin_probe.shape[0]
print(cant)
show(admin_probe,20)

In [ ]:
sqli = df[df["attack_patterns"].apply(lambda x: "sqli" in x)]
show(sqli,10)

In [ ]:
sqli = df[df["attack_patterns"].apply(lambda x: "sqli" in x) & (df['anomaly']!=1)]
cant=sqli.shape[0]
print(cant)
show(sqli,20)

In [12]:
df.loc[
    df['attack_patterns'].apply(lambda x: 'sqli' in x) & (df['anomaly'] == -1),
    'anomaly'
] = 1

In [14]:
df['anomaly'].value_counts()

anomaly
 0    492237
-1    475401
 1     86968
Name: count, dtype: int64

In [15]:
df.to_csv(
    'access_log_master_manual_lab_if_score.csv',
    index=False
)

## 3. Severidad usando Isolation Forest

In [16]:
q01 = df['if_score'].quantile(0.01)
q03 = df['if_score'].quantile(0.03)

def anomaly_severity(score):
    if score <= q01:
        return 'critical'
    elif score <= q03:
        return 'high'
    return 'medium'

df['severity'] = df['if_score'].apply(anomaly_severity)

df[['if_score', 'severity']].head()

,if_score,severity
0,0.042052,medium
1,0.030259,medium
2,0.041807,medium
3,0.029720,medium
4,0.045362,medium


In [28]:

df_sig=df[df["signature_flag"]==1].copy()
df_sig.to_csv("acces_log_confirmed_attack",index=False)

##  Etiquetado supervisasdo

In [ ]:
def manual_attack_verification(df, index, log: dict):
    """
    Muestra el log y permite etiquetar inmediatamente.
    Si el usuario pulsa 'A', se detiene el proceso.
    """

    print("\n" + "=" * 120)
    print(f"IP:        {log.get('ip_client')}")
    print(f"Timestamp: {log.get('timestamp')}")
    print(f"Method:    {log.get('method')}   Status: {log.get('status')}")
    print(f"URL:\n{log.get('url')}")
    print(f"User-Agent:\n{log.get('user_agent')}")
    print(f"Anomaly:  {log.get('anomaly')}")
    print(f"Severity:  {log.get('severity')}")
    print(f"IF_pred:   {log.get('if_prediction')}")
    print(f"Patterns:  {log.get('patterns_detectados')}")
    print("=" * 120)

    while True:
        decision = input("¿Es un ataque REAL? (1 = sí, 0 = no, A = abortar): ").strip().upper()

        if decision in ("0", "1"):
            df.at[index, "anomaly"] = int(decision)   # ← actualización inmediata
            return True   # continuar

        if decision == "A":
            print("\n🛑 Etiquetado detenido por el usuario.")
            return False  # detener

        print("Entrada inválida. Usa 1, 0 o A.")


# ---- PROCESO PRINCIPAL ----

subset = df[
    (df["severity"] == "critical") &
    (df["if_prediction"] == -1) &
    (df["anomaly"] == -1)
]

for index, row in subset.iterrows():
    continuar = manual_attack_verification(df, index, row.to_dict())
    if not continuar:
        break



In [18]:
df.to_csv(
    'access_log_master_manual_lab_if_score.csv',
    index=False
)

## 4. Etiquetado final
/s/scriptorium/item?Search=&__waf_test__
/s/scriptorium/item?Search=&page=72%27EXTRACTVALUE%289704%2CCONCAT%280x7e%2C%28SELECT%2F%2A%2A%2F%28ELT%289704%3D9704%2C1%29%29%29%2C0x7e%29%29--+-&property%5B0%5D%5Bproperty%5D=63&property%5B0%5D%5Btext%5D=Tesis+de+Maestria&property%5B0%5D%5Btype%5D=eq&sort_by=resource_class_label&sort_order=desc


In [16]:
def final_label(row):
    if row['signature_flag'] == 1:
        return 1  # Ataque confirmado
    if row['if_prediction'] == -1 and row['severity'] in ['critical', 'high']:
        return -1  # Ataque probable
    if row['if_prediction'] == -1:
        return 3  # Sospechoso / Recon
    return 0  # Normal

df['attack_label'] = df.apply(final_label, axis=1)

df['attack_label'].value_counts()



attack_label
 0    985051
 1     42449
-1     26853
Name: count, dtype: int64

In [ ]:
critical= df[df['severity'] == "critical"].copy()
df['severity'].value_counts()


severity
medium      1020185
high          23621
critical      10547
Name: count, dtype: int64

In [ ]:
def show2(df_review,num):
    labels = {}
    i=0
    for idx, row in df_review.iterrows():
        i=i+1
        print("="*120)
        print(f"Index: {idx}")
        print(f"IP: {row['ip_client']}")
        print(f"Timestamp: {row['timestamp']}")
        print(f"Method: {row['method']}  Status: {row['status']}")
        print(f"URL:\n{row['url']}")
        print(f"User-Agent:\n{row['user_agent']}")

        if(i==num):break
        


df0=pd.read_csv('../data/target/access_log_master_manual_labeling.csv')
at= df0[df0['anomaly'] == 1].copy()
show2(at,10)

In [30]:
at.to_csv("acces_log_confirmed_attack_manual",index=False)

In [ ]:
to_review = df[df['attack_label'] == -1].copy()

cols_to_show = [
    'ip_client',
    'timestamp',
    'method',
    'status',
    'url',
    'user_agent',
    'if_score',
    'severity',
    'attack_patterns'
]

to_review[cols_to_show].head(10)


In [10]:
def manual_labeling(df_review):
    labels = {}
    
    for idx, row in df_review.iterrows():
        print("="*120)
        print(f"Index: {idx}")
        print(f"IP: {row['ip_client']}")
        print(f"Timestamp: {row['timestamp']}")
        print(f"Method: {row['method']}  Status: {row['status']}")
        print(f"URL:\n{row['url']}")
        print(f"User-Agent:\n{row['user_agent']}")
        print(f"IF score: {row['if_score']}  Severity: {row['severity']}")
        print(f"Patterns detectados: {row['attack_patterns']}")
        print("\nEtiqueta:")
        print("1 = Ataque confirmado")
        print("3 = Sospechoso / Recon")
        print("0 = Normal")
        
        label = input("Tu decisión: ")
        labels[idx] = int(label)
    
    return labels


In [ ]:
manual_decisions = manual_labeling(to_review)


Index: 31
IP: 34.96.45.206
Timestamp: 2025-11-25 00:00:29-05:00
Method: GET  Status: 200
URL:
/wp-content/uploads/2022/10/libro-de-graduados-2012-2013.pdf
User-Agent:
Mozilla/5.0 (Linux; Android 6.0.1; Nexus 5X Build/MMB29P) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)
IF score: -0.0081887421487543  Severity: high
Patterns detectados: []

Etiqueta:
1 = Ataque confirmado
3 = Sospechoso / Recon
0 = Normal


## 5. Correcciones manuales (whitelisting)

In [ ]:
whitelist_patterns = [
    r"/theme/yui_combo.php",
    r"/static/",
    r"/assets/"
]

for p in whitelist_patterns:
    df.loc[df['url'].str.contains(p, regex=True, na=False), 'attack_label'] = 0

df['attack_label'].value_counts()

## 6. Validación básica

In [ ]:
df.groupby('attack_label').size() / len(df)

## 7. Exportar dataset etiquetado

In [ ]:
df.to_csv('labeled_web_logs_final.csv', index=False)
df.head()